In [1]:
import json
from pathlib import Path
from typing import List, Dict
def gene_to_searchable_text(gene: Dict, article_title: str) -> str:
    """将基因信息转换为可检索的文本"""
    parts = []

    # 核心字段
    if gene.get("Gene_Name"):
        parts.append(f"基因名: {gene['Gene_Name']}")
    if gene.get("Species_Latin_Name"):
        parts.append(f"物种: {gene['Species_Latin_Name']}")
    if gene.get("Protein_Family_or_Domain"):
        parts.append(f"蛋白家族/结构域: {gene['Protein_Family_or_Domain']}")

    # 功能相关
    if gene.get("Core_Phenotypic_Effect"):
        parts.append(f"核心表型效应: {gene['Core_Phenotypic_Effect']}")
    if gene.get("Regulatory_Mechanism"):
        parts.append(f"调控机制: {gene['Regulatory_Mechanism']}")
    if gene.get("Regulatory_Pathway"):
        parts.append(f"调控通路: {gene['Regulatory_Pathway']}")
    if gene.get("Biosynthetic_Pathway"):
        parts.append(f"生物合成通路: {gene['Biosynthetic_Pathway']}")

    # 互作和调控
    if gene.get("Interacting_Proteins"):
        parts.append(f"互作蛋白: {gene['Interacting_Proteins']}")
    if gene.get("Upstream_or_Downstream_Regulation"):
        parts.append(f"上下游调控: {gene['Upstream_or_Downstream_Regulation']}")

    # 研究信息
    if gene.get("Research_Topic"):
        parts.append(f"研究主题: {gene['Research_Topic']}")
    if gene.get("Trait_Category"):
        parts.append(f"性状类别: {gene['Trait_Category']}")
    if gene.get("Key_Environment_or_Treatment_Factor"):
        parts.append(f"关键环境/处理因素: {gene['Key_Environment_or_Treatment_Factor']}")

    # 实验验证
    if gene.get("Core_Validation_Method"):
        parts.append(f"核心验证方法: {gene['Core_Validation_Method']}")
    if gene.get("Experimental_Methods"):
        parts.append(f"实验方法: {gene['Experimental_Methods']}")

    # 育种价值
    if gene.get("Breeding_Application_Value"):
        parts.append(f"育种应用价值: {gene['Breeding_Application_Value']}")

    # 关键发现
    if gene.get("Summary_key_Findings_of_Core_Gene"):
        parts.append(f"关键发现: {gene['Summary_key_Findings_of_Core_Gene']}")

    return "\n".join(parts)

In [2]:
# 示例：从某篇玉米抗旱研究论文中提取的基因信息
gene_data = {
    "Gene_Name": "ZmDREB2A",
    "Species_Latin_Name": "Zea mays",
    "Protein_Family_or_Domain": "AP2/EREBP transcription factor family",
    "Core_Phenotypic_Effect": "增强玉米幼苗抗旱性",
    "Regulatory_Mechanism": "通过ABA依赖途径调控下游基因表达",
    "Regulatory_Pathway": "ABA信号转导通路",
    "Biosynthetic_Pathway": None,
    "Interacting_Proteins": "ZmABF3, ZmNAC48",
    "Upstream_or_Downstream_Regulation": "上游受ZmSnRK2.8激酶激活，下游调控ZmRD29等基因",
    "Research_Topic": "玉米抗旱分子机制",
    "Trait_Category": "非生物胁迫抗性",
    "Key_Environment_or_Treatment_Factor": "干旱胁迫（20% PEG6000处理）",
    "Core_Validation_Method": "转基因验证、基因敲除",
    "Experimental_Methods": "qRT-PCR, Western blot, EMSA, 酵母单杂交",
    "Breeding_Application_Value": "可用于培育抗旱玉米新品种",
    "Summary_key_Findings_of_Core_Gene": "ZmDREB2A过表达显著提高玉米抗旱性30%以上"
}

article_title = "玉米转录因子ZmDREB2A调控抗旱性的分子机制研究"
print(gene_to_searchable_text(gene_data, article_title))

基因名: ZmDREB2A
物种: Zea mays
蛋白家族/结构域: AP2/EREBP transcription factor family
核心表型效应: 增强玉米幼苗抗旱性
调控机制: 通过ABA依赖途径调控下游基因表达
调控通路: ABA信号转导通路
互作蛋白: ZmABF3, ZmNAC48
上下游调控: 上游受ZmSnRK2.8激酶激活，下游调控ZmRD29等基因
研究主题: 玉米抗旱分子机制
性状类别: 非生物胁迫抗性
关键环境/处理因素: 干旱胁迫（20% PEG6000处理）
核心验证方法: 转基因验证、基因敲除
实验方法: qRT-PCR, Western blot, EMSA, 酵母单杂交
育种应用价值: 可用于培育抗旱玉米新品种
关键发现: ZmDREB2A过表达显著提高玉米抗旱性30%以上


In [8]:
import json
from pathlib import Path
from typing import List, Dict
def load_json_files(data_dir: Path) -> List[Dict]:
    """加载所有JSON文件"""
    files = list(data_dir.glob("*.json"))
    data = []
    for f in files:
        try:
            with open(f, "r", encoding="utf-8") as fp:
                data.append(json.load(fp))
        except Exception as e:
            print(f"Error loading {f}: {e}")
    return data
a=load_json_files(Path("../data/data2"))
for item in a:
    item=gene_to_searchable_text(item, item.get("Article_Title", ""))
    print(item)

In [9]:
from dataclasses import dataclass
import json
from pathlib import Path
from typing import List, Dict, Any
@dataclass
class GeneChunk:
    """基因信息切片，用于检索"""
    chunk_id: str
    text: str                # 用于检索的文本
    gene_name: str           # 基因名
    article_title: str       # 文章标题
    journal: str
    doi: str
    species: str
    category: str            # Plant/Animal/Microbial
    metadata: Dict[str, Any] # 完整的基因字段

In [10]:
def process_all_data(data_dir: Path) -> List[GeneChunk]:
    """处理所有数据，生成检索用的切片"""
    all_chunks = []
    raw_data = load_json_files(data_dir)

    for doc in raw_data:
        title = doc.get("Title", "Unknown")
        journal = doc.get("Journal", "Unknown")
        doi = doc.get("DOI", "")

        # 处理各类基因
        for category, key in [("Plant", "Plant_Genes"),
                              ("Animal", "Animal_Genes"),
                              ("Microbial", "Microbial_Genes")]:
            genes = doc.get(key, [])
            for i, gene in enumerate(genes):
                gene_name = gene.get("Gene_Name", f"Unknown_{i}")
                species = gene.get("Species_Latin_Name", gene.get("Species", "Unknown"))

                # 生成可检索文本
                text = gene_to_searchable_text(gene, title)

                chunk = GeneChunk(
                    chunk_id=f"{doi}_{gene_name}_{i}",
                    text=text,
                    gene_name=gene_name,
                    article_title=title,
                    journal=journal,
                    doi=doi,
                    species=species,
                    category=category,
                    metadata=gene
                )
                all_chunks.append(chunk)

    print(f"Processed {len(raw_data)} articles, {len(all_chunks)} gene chunks")
    return all_chunks

In [14]:
from pathlib import Path

# 1. 调用函数
data_dir = Path("/data/haotianwu/biojson/data/data2")
chunks = process_all_data(data_dir)
print(chunks)

Processed 2 articles, 4 gene chunks
[GeneChunk(chunk_id='10.1186/s12864-021-08255-0_ShCOI1-4_0', text='基因名: ShCOI1-4\n物种: Saccharum spp. hybrid\n蛋白家族/结构域: COI1 family F-box protein with F-box domain, Transp_inhibit (transport inhibitor response 1) domain, and AMN1 leucine-rich repeat (LRR) domain.\n核心表型效应: Hormone- and pathogen-responsive COI1-like gene whose expression is activated by ABA, SA, and smut infection and suppressed by cold and drought, suggesting involvement in sugarcane defense and abiotic stress signaling.\n调控机制: Transcriptionally up-regulated by ABA and SA and down-regulated by cold and PEG-induced drought; induction pattern under smut infection differs among cultivars, with stronger responses in some resistant lines.\n调控通路: COI1-related jasmonate signaling module with cross-talk to ABA- and SA-mediated stress-response pathways in sugarcane.\n研究主题: Hormone-responsive and stress-responsive regulation of COI1 genes in sugarcane, focusing on roles in biotic (smut disease) 

In [15]:
from pathlib import Path
print("\n" + "="*80)
print(f"数据处理完成！共生成 {len(chunks)} 个基因切片")
print("="*80)

for i, chunk in enumerate(chunks, 1):
    print(f"\n{'─'*80}")
    print(f"【切片 {i}/{len(chunks)}】")
    print(f"{'─'*80}")
    
    # 基本信息
    print(f"🧬 基因名:    {chunk.gene_name}")
    print(f"🌿 物种:      {chunk.species}")
    print(f"📁 类别:      {chunk.category}")
    print(f"📄 文献:      {chunk.article_title}")
    print(f"📚 期刊:      {chunk.journal}")
    print(f"🔗 DOI:       {chunk.doi}")
    print(f"🆔 Chunk ID:  {chunk.chunk_id}")
    
    # 可检索文本（只显示前3行）
    print(f"\n📝 可检索内容预览:")
    print("┌" + "─"*78 + "┐")
    for line in chunk.text.split('\n')[:3]:
        print(f"│ {line:<76} │")
    print(f"│ ... (共 {len(chunk.text.split(chr(10)))} 行) {'':<60} │")
    print("└" + "─"*78 + "┘")

print("\n" + "="*80 + "\n")


数据处理完成！共生成 4 个基因切片

────────────────────────────────────────────────────────────────────────────────
【切片 1/4】
────────────────────────────────────────────────────────────────────────────────
🧬 基因名:    ShCOI1-4
🌿 物种:      Saccharum spp. hybrid
📁 类别:      Plant
📄 文献:      Genome-wide identification and expression analysis of the coronatine-insensitive 1 (COI1) gene family in response to biotic and abiotic stresses in Saccharum
📚 期刊:      BMC Genomics
🔗 DOI:       10.1186/s12864-021-08255-0
🆔 Chunk ID:  10.1186/s12864-021-08255-0_ShCOI1-4_0

📝 可检索内容预览:
┌──────────────────────────────────────────────────────────────────────────────┐
│ 基因名: ShCOI1-4                                                                │
│ 物种: Saccharum spp. hybrid                                                    │
│ 蛋白家族/结构域: COI1 family F-box protein with F-box domain, Transp_inhibit (transport inhibitor response 1) domain, and AMN1 leucine-rich repeat (LRR) domain. │
│ ... (共 16 行)                            